# CyberGYM 500M Training — Kaggle T4
Run Cell 1 first. If it prints DIAGNOSTIC PASSED, run Cell 2.


In [ ]:
import os, hashlib, subprocess, sys

# --- clone ---
if not os.path.isdir("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "https://github.com/ziyy1337/train-llm-from-scratch", "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print("CLONE OK")

# --- install deps ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jaxtyping", "beartype"], check=True)
print("DEPS OK")

# --- find data files anywhere under /kaggle/input ---
def find(name):
    for root, _, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return None
train_path = find("cyber_train.h5")
dev_path = find("cyber_dev.h5")
print("train_path:", train_path)
print("dev_path:", dev_path)
assert train_path and dev_path, "DATA FILES NOT FOUND - upload cyber_train.h5 and cyber_dev.h5 to Kaggle first"

# --- hash check ---
expected = {
    "cyber_train.h5": "1564507c40e53863cddcea738d9bec0fd5e4ad940e38aefe8a3ccc92746ff9eb",
    "cyber_dev.h5": "a997a1b2f7ea5ebb53fea42d6ed39544e21bec10bb1ae09e76a248ab69373db3",
}
for p in (train_path, dev_path):
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    name = os.path.basename(p)
    print(f"{name}: size={os.path.getsize(p)} sha256={h} {'MATCH' if h == expected[name] else 'MISMATCH'}")
    assert h == expected[name], f"{name} hash mismatch - re-upload the file"

with open("/kaggle/working/paths.txt", "w") as f:
    f.write(f"{train_path}\n{dev_path}\n")
print("DATA VERIFIED OK")

# --- 1-step diagnostic ---
r = subprocess.run([
    "python", "scripts/train_transformer.py", "--preset", "500m", "--arch", "modern",
    "--device", "cuda", "--amp-dtype", "fp16", "--steps", "1", "--eval-every", "10",
    "--batch-size", "2", "--checkpoint-every", "0", "--set", "t_eval_iters=1",
    "--train-path", train_path, "--dev-path", dev_path,
], capture_output=True, text=True)
print("DIAG EXIT:", r.returncode)
print("DIAG STDOUT (tail):\n", r.stdout[-1500:])
print("DIAG STDERR (tail):\n", r.stderr[-1500:])
assert r.returncode == 0, "1-step diagnostic FAILED - do not run Cell 2"
print("DIAGNOSTIC PASSED - safe to run Cell 2")



In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.chdir("/kaggle/working/repo")
tp, dp = open("/kaggle/working/paths.txt").read().splitlines()
!python scripts/train_transformer.py --preset 500m --arch modern --device cuda --amp-dtype fp16 --steps 15000 --eval-every 1000 --batch-size 2 --grad-accum 2 --checkpoint-every 500 --set t_eval_iters=20 --train-path {tp} --dev-path {dp} --set t_out_path=/kaggle/working/models/500m_cybergym.pt

